# Mr.X R-GNN - Stage X2 PPO

Notebook Kaggle per continuare Mr.X con PPO partendo dal warm-start BC.

Setup:
- Mr.X policy/value: checkpoint registry `mrx_bc_v*.pt` o artifact legacy `mrx_rgnn_bc_best_*.pt`;
- detectives: best GNN PPO registry `detective_ppo_v*.pt` o artifact legacy `rgnn_ppo_best_*.pt`, congelata;
- rollout: sampling dalla categorical sulle sole azioni legali `(destination, ticket)`;
- eval: argmax della policy Mr.X contro detectives GNN frozen;
- checkpoint selection: best validation score, non ultimo update.

Output in `/kaggle/working/mrx_ppo_checkpoints`:
- `mrx_rgnn_ppo_best_*.pt`
- `mrx_rgnn_ppo_last_*.pt`
- `mrx_rgnn_ppo_history_*.csv`
- `mrx_rgnn_ppo_curves_*.png`

Se il candidato passa la validation large e la promotion gate, copialo localmente come `Notebook/Models/mrx/mrx_ppo_vXXX.pt` e aggiorna `Notebook/Registry/`, senza sovrascrivere i vecchi best.

## 1. Setup repo

In [ ]:
%%bash
set -e
cd /kaggle/working
python -m pip install -q networkx numpy pandas tqdm scipy matplotlib torch || true

REPO_DIR=/kaggle/working/scotland_yard
if [ ! -d "$REPO_DIR" ]; then
  CANDIDATE=$(find /kaggle/input -maxdepth 4 -type d -name scotland_yard | head -n 1 || true)
  if [ -n "$CANDIDATE" ]; then
    echo "Copying repo from Kaggle input: $CANDIDATE"
    cp -R "$CANDIDATE" "$REPO_DIR"
  else
    echo "Cloning repo from GitHub"
    git clone --depth 1 https://github.com/Jacopo888/scotland_yard.git "$REPO_DIR"
  fi
fi
mkdir -p /kaggle/working/mrx_ppo_checkpoints
ls "$REPO_DIR"


## 2. Locate checkpoints and imports

In [ ]:
import glob
import json
import math
import os
import random
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.distributions import Categorical

REPO_DIR = Path('/kaggle/working/scotland_yard')
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from detective_engine import DetectiveEngine
from game import Game
from gnn_detective_engine import GNNDetectiveEngine
from gnn_mrx_engine import GNNMrXEngine, RELATIONS, N_NODES
from utility import _min_detective_distance


def choose_checkpoint(candidates, preferred_tokens):
    unique = sorted(set(candidates), key=lambda p: os.path.basename(p))
    for token in preferred_tokens:
        matches = [p for p in unique if token in os.path.basename(p)]
        if matches:
            return matches[-1]
    return unique[-1] if unique else None


mrx_candidates = []
mrx_candidates += sorted(glob.glob('/kaggle/input/**/mrx_bc_*.pt', recursive=True))
mrx_candidates += sorted(glob.glob('/kaggle/input/**/mrx_rgnn_bc_best_*.pt', recursive=True))
mrx_candidates += sorted(glob.glob(str(REPO_DIR / 'Notebook' / 'Models' / 'mrx' / 'mrx_bc_*.pt')))
mrx_candidates += sorted(glob.glob(str(REPO_DIR / 'Notebook' / 'Models' / 'mrx' / 'mrx_rgnn_bc_best_*.pt')))
mrx_candidates += sorted(glob.glob(str(REPO_DIR / 'Notebook' / 'mrx_rgnn_bc_best_*.pt')))
MRX_BC_CKPT = choose_checkpoint(mrx_candidates, ('mrx_bc_', 'mrx_rgnn_bc_best_'))
assert MRX_BC_CKPT, 'Missing warm-start Mr.X checkpoint mrx_bc_*.pt or mrx_rgnn_bc_best_*.pt.'

det_candidates = []
det_candidates += sorted(glob.glob('/kaggle/input/**/detective_ppo_*.pt', recursive=True))
det_candidates += sorted(glob.glob('/kaggle/input/**/rgnn_ppo_best_*.pt', recursive=True))
det_candidates += sorted(glob.glob(str(REPO_DIR / 'Notebook' / 'Models' / 'detectives' / 'detective_ppo_*.pt')))
det_candidates += sorted(glob.glob(str(REPO_DIR / 'Notebook' / 'Models' / 'detectives' / 'rgnn_ppo_best_*.pt')))
det_candidates += sorted(glob.glob(str(REPO_DIR / 'Notebook' / 'rgnn_ppo_best_*.pt')))
DETECTIVE_CKPT = choose_checkpoint(det_candidates, ('detective_ppo_', 'rgnn_ppo_best_'))
assert DETECTIVE_CKPT, 'Missing detective checkpoint detective_ppo_*.pt or rgnn_ppo_best_*.pt.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Mr.X BC checkpoint:', MRX_BC_CKPT)
print('Detective frozen checkpoint:', DETECTIVE_CKPT)

## 3. Hyperparameters

In [ ]:
SEED = 20260521
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = Path('/kaggle/working/mrx_ppo_checkpoints')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Conservative first PPO continuation. Scale these after the curve looks sane.
UPDATES = 30
GAMES_PER_UPDATE = 16
PPO_EPOCHS = 2
MINIBATCH_SIZE = 128
LR = 5e-5
WEIGHT_DECAY = 1e-5
GAMMA = 0.95
CLIP_EPS = 0.15
VALUE_COEF = 0.25
ENTROPY_COEF = 0.005
GRAD_CLIP = 1.0
TARGET_KL = 0.03

EVAL_EVERY = 3
EVAL_GAMES = 100
INITIAL_EVAL_GAMES = 100

# Same reward family used by the logger, kept intentionally simple.
R_TIMEOUT = +10.0
R_CAPTURE = -10.0
R_STEP = +0.05
R_DIST_COEF = +0.10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Run tag:', RUN_TAG)


## 4. Load frozen detectives and warm-start Mr.X

In [ ]:
detective_policy = GNNDetectiveEngine(checkpoint_path=DETECTIVE_CKPT, device=DEVICE)
mrx_policy = GNNMrXEngine(checkpoint_path=MRX_BC_CKPT, device=DEVICE)
model = mrx_policy.model
dense_adj = mrx_policy.dense_adj
node_static = mrx_policy.node_static

bc_ckpt = torch.load(MRX_BC_CKPT, map_location=mrx_policy.device, weights_only=False)
base_config = dict(bc_ckpt.get('config', {}))
return_mean = float(base_config.get('return_mean', mrx_policy.return_mean))
return_std = float(base_config.get('return_std', mrx_policy.return_std))

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print('Detective metadata:', detective_policy.metadata)
print('Mr.X BC metadata:', mrx_policy.metadata)
print('Return norm mean/std:', return_mean, return_std)


## 5. Environment helpers

In [ ]:
N_REL = len(RELATIONS)


def decode_action(action):
    destination_idx = int(action) // N_REL
    ticket_idx = int(action) % N_REL
    return str(destination_idx + 1), RELATIONS[ticket_idx]


def play_detective_phase(game, engine):
    for detective_id in range(game.num_detectives):
        detective_policy.play_detective_turn(game, engine.belief_state, detective_id)
        if game.check_victory(silent=True):
            return True
        engine.kalman_filter()
    game.detectives_moves.append(game.detectives_pos[:])
    return False


def collate_samples(samples):
    return {
        'node_dyn': torch.from_numpy(np.stack([s['node_dyn'] for s in samples])).float().to(mrx_policy.device),
        'glob': torch.from_numpy(np.stack([s['glob'] for s in samples])).float().to(mrx_policy.device),
        'mrx_pos_idx': torch.tensor([s['mrx_pos_idx'] for s in samples], dtype=torch.long, device=mrx_policy.device),
        'legal_mask': torch.from_numpy(np.stack([s['legal_mask'] for s in samples])).bool().to(mrx_policy.device),
    }


@torch.no_grad()
def sample_mrx_action(game, engine):
    sample = mrx_policy.build_input(game, engine.belief_state)
    if not sample['legal_mask'].any():
        return sample, None, None, None, None
    batch = collate_samples([sample])
    logits, value = model(batch, dense_adj, node_static)
    masked_logits = logits.masked_fill(~batch['legal_mask'], -1e9)
    dist = Categorical(logits=masked_logits)
    action = dist.sample()
    log_prob = dist.log_prob(action)
    entropy = dist.entropy()
    return (
        sample,
        int(action.item()),
        float(log_prob.item()),
        float(value[0].item()),
        float(entropy.item()),
    )


def add_returns_and_advantages(transitions):
    running = 0.0
    for t in reversed(transitions):
        if t['done']:
            running = 0.0
        running = float(t['reward']) + GAMMA * running
        t['return_raw'] = running
        t['return_norm'] = (running - return_mean) / max(return_std, 1e-6)
        t['advantage'] = t['return_norm'] - t['value']
    return transitions


def collect_episode(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    model.eval()
    detective_policy.reset()
    mrx_policy.reset()
    game = Game()
    engine = DetectiveEngine(game.detectives_pos)
    transitions = []
    blocked = 0

    while True:
        if game.check_victory(silent=True):
            break

        if play_detective_phase(game, engine):
            if transitions:
                transitions[-1]['reward'] += R_CAPTURE
                transitions[-1]['done'] = True
            break

        sample, action, old_log_prob, value, entropy = sample_mrx_action(game, engine)
        if action is None:
            blocked += 1
            game.x_automated_turn(game.mrx_pos, None)
            if game.check_victory(silent=True):
                if transitions:
                    transitions[-1]['reward'] += R_TIMEOUT if game.winner == 1 else R_CAPTURE
                    transitions[-1]['done'] = True
                break
            engine.update_belief_after_mrx_move(None)
            if (game.turn - 3) % 5 == 0:
                engine.mrx_is_spotted(game.mrx_pos)
            detective_policy.observe_mrx_move(game, None)
            mrx_policy.observe_mrx_move(game, None)
            continue

        destination, ticket = decode_action(action)
        legal_before = game.find_legal_moves_x()
        assert destination in legal_before.get(ticket, []), (destination, ticket, legal_before)

        game.x_automated_turn(destination, ticket)
        min_dist = _min_detective_distance(game.detectives_pos, game.mrx_pos)
        reward = R_STEP + R_DIST_COEF * min_dist
        done = False

        if game.check_victory(silent=True):
            reward += R_TIMEOUT if game.winner == 1 else R_CAPTURE
            done = True

        transitions.append({
            'sample': sample,
            'action': int(action),
            'old_log_prob': float(old_log_prob),
            'value': float(value),
            'entropy': float(entropy),
            'reward': float(reward),
            'done': bool(done),
            'ticket': ticket,
            'min_dist': int(min_dist),
        })

        if done:
            break

        engine.update_belief_after_mrx_move(ticket)
        if (game.turn - 3) % 5 == 0:
            engine.mrx_is_spotted(game.mrx_pos)
        detective_policy.observe_mrx_move(game, ticket)
        mrx_policy.observe_mrx_move(game, ticket)

    add_returns_and_advantages(transitions)
    return transitions, {
        'winner': int(game.winner),
        'mrx_win': int(game.winner == 1),
        'final_turn': int(game.turn),
        'blocked': int(blocked),
    }


## 6. PPO update and evaluation

In [ ]:
def build_buffer(transitions):
    adv = np.array([t['advantage'] for t in transitions], dtype=np.float32)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    return {
        'samples': [t['sample'] for t in transitions],
        'actions': torch.tensor([t['action'] for t in transitions], dtype=torch.long, device=mrx_policy.device),
        'old_log_probs': torch.tensor([t['old_log_prob'] for t in transitions], dtype=torch.float32, device=mrx_policy.device),
        'returns': torch.tensor([t['return_norm'] for t in transitions], dtype=torch.float32, device=mrx_policy.device),
        'advantages': torch.from_numpy(adv).to(mrx_policy.device),
    }


def ppo_update(buffer):
    n = len(buffer['actions'])
    metrics = defaultdict(float)
    steps = 0
    stop_early = False

    for epoch in range(PPO_EPOCHS):
        order = torch.randperm(n, device=mrx_policy.device)
        for start in range(0, n, MINIBATCH_SIZE):
            idx = order[start:start + MINIBATCH_SIZE]
            samples = [buffer['samples'][int(i)] for i in idx.detach().cpu().tolist()]
            batch = collate_samples(samples)
            actions = buffer['actions'][idx]
            old_log_probs = buffer['old_log_probs'][idx]
            returns = buffer['returns'][idx]
            advantages = buffer['advantages'][idx]

            model.train()
            logits, values = model(batch, dense_adj, node_static)
            masked_logits = logits.masked_fill(~batch['legal_mask'], -1e9)
            dist = Categorical(logits=masked_logits)
            log_probs = dist.log_prob(actions)
            entropy = dist.entropy().mean()

            ratio = torch.exp(log_probs - old_log_probs)
            unclipped = ratio * advantages
            clipped = torch.clamp(ratio, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS) * advantages
            policy_loss = -torch.min(unclipped, clipped).mean()
            value_loss = F.mse_loss(values, returns)
            loss = policy_loss + VALUE_COEF * value_loss - ENTROPY_COEF * entropy

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            with torch.no_grad():
                approx_kl = (old_log_probs - log_probs).mean().item()
                clipfrac = ((ratio - 1.0).abs() > CLIP_EPS).float().mean().item()
                pred = masked_logits.argmax(dim=-1)
                acc = (pred == actions).float().mean().item()

            bs = len(actions)
            steps += bs
            metrics['loss'] += float(loss.item()) * bs
            metrics['policy_loss'] += float(policy_loss.item()) * bs
            metrics['value_loss'] += float(value_loss.item()) * bs
            metrics['entropy'] += float(entropy.item()) * bs
            metrics['approx_kl'] += float(approx_kl) * bs
            metrics['clipfrac'] += float(clipfrac) * bs
            metrics['argmax_acc_on_rollout'] += float(acc) * bs

            if approx_kl > TARGET_KL:
                stop_early = True
                break
        if stop_early:
            break

    out = {k: v / max(steps, 1) for k, v in metrics.items()}
    out['ppo_steps'] = int(steps)
    out['early_stop_kl'] = bool(stop_early)
    return out


@torch.no_grad()
def eval_argmax(n_games, seed_offset):
    model.eval()
    wins = 0
    turns = []
    illegal = 0
    tickets = Counter()
    dists = []

    for i in range(n_games):
        random.seed(seed_offset + i)
        np.random.seed(seed_offset + i)
        detective_policy.reset()
        mrx_policy.reset()
        game = Game()
        engine = DetectiveEngine(game.detectives_pos)

        while True:
            if game.check_victory(silent=True):
                break
            if play_detective_phase(game, engine):
                break

            legal_before = game.find_legal_moves_x()
            destination, ticket = mrx_policy.choose_action(game, engine.belief_state)
            if ticket is not None and destination not in legal_before.get(ticket, []):
                illegal += 1
            game.x_automated_turn(destination, ticket)
            tickets.update([ticket if ticket is not None else 'blocked'])
            dists.append(_min_detective_distance(game.detectives_pos, game.mrx_pos))

            if game.check_victory(silent=True):
                break

            engine.update_belief_after_mrx_move(ticket)
            if (game.turn - 3) % 5 == 0:
                engine.mrx_is_spotted(game.mrx_pos)
            detective_policy.observe_mrx_move(game, ticket)
            mrx_policy.observe_mrx_move(game, ticket)

        wins += int(game.winner == 1)
        turns.append(int(game.turn))

    wr = wins / n_games if n_games else 0.0
    avg_turn = float(np.mean(turns)) if turns else 0.0
    score = wr * 100.0 + avg_turn - illegal * 100.0
    return {
        'n_games': int(n_games),
        'mrx_wins': int(wins),
        'mrx_winrate': float(wr),
        'avg_final_turn': avg_turn,
        'illegal_actions': int(illegal),
        'ticket_counts': {str(k): int(v) for k, v in tickets.items()},
        'avg_min_dist_after_mrx': float(np.mean(dists)) if dists else None,
        'validation_score': float(score),
    }


## 7. Train loop

In [ ]:
best_path = OUT_DIR / f'mrx_rgnn_ppo_best_{RUN_TAG}.pt'
last_path = OUT_DIR / f'mrx_rgnn_ppo_last_{RUN_TAG}.pt'
history_path = OUT_DIR / f'mrx_rgnn_ppo_history_{RUN_TAG}.csv'


def checkpoint_payload(update, eval_result, history):
    cfg = dict(base_config)
    cfg.update({
        'ppo': True,
        'gamma': GAMMA,
        'clip_eps': CLIP_EPS,
        'value_coef': VALUE_COEF,
        'entropy_coef': ENTROPY_COEF,
        'lr': LR,
    })
    return {
        'model_state_dict': model.state_dict(),
        'config': cfg,
        'update': int(update),
        'mrx_wr': float(eval_result['mrx_winrate']),
        'gnn_wr': float(eval_result['mrx_winrate']),
        'validation_score': float(eval_result['validation_score']),
        'eval': eval_result,
        'source_kind': 'mrx_bc',
        'source_checkpoint': MRX_BC_CKPT,
        'detective_checkpoint': DETECTIVE_CKPT,
        'run_tag': RUN_TAG,
        'history_tail': history[-10:],
    }


history = []
best_eval = eval_argmax(INITIAL_EVAL_GAMES, seed_offset=SEED + 500000)
best_score = best_eval['validation_score']
torch.save(checkpoint_payload(0, best_eval, history), best_path)
print('Initial eval:', json.dumps(best_eval, indent=2))
print('Initial best saved:', best_path)

started = time.time()
for update in range(1, UPDATES + 1):
    transitions = []
    episode_stats = []
    for g in tqdm(range(GAMES_PER_UPDATE), desc=f'rollout {update:03d}', leave=False):
        ts, st = collect_episode(SEED + update * 10000 + g)
        transitions.extend(ts)
        episode_stats.append(st)

    if not transitions:
        print(f'update {update:03d}: no transitions collected')
        continue

    buffer = build_buffer(transitions)
    train_metrics = ppo_update(buffer)
    rollout_wr = float(np.mean([s['mrx_win'] for s in episode_stats]))
    rollout_turn = float(np.mean([s['final_turn'] for s in episode_stats]))
    rollout_entropy = float(np.mean([t['entropy'] for t in transitions]))
    ticket_counts = Counter(t['ticket'] for t in transitions)

    eval_result = None
    if update % EVAL_EVERY == 0 or update == UPDATES:
        eval_result = eval_argmax(EVAL_GAMES, seed_offset=SEED + 700000 + update * 1000)
        if eval_result['validation_score'] > best_score:
            best_score = eval_result['validation_score']
            torch.save(checkpoint_payload(update, eval_result, history), best_path)
            print('  saved best:', best_path)

    row = {
        'update': int(update),
        'n_transitions': int(len(transitions)),
        'rollout_mrx_wr': rollout_wr,
        'rollout_avg_turn': rollout_turn,
        'rollout_entropy': rollout_entropy,
        'rollout_ticket_counts': dict(ticket_counts),
        **{f'train_{k}': v for k, v in train_metrics.items()},
    }
    if eval_result is not None:
        row.update({f'eval_{k}': v for k, v in eval_result.items() if k != 'ticket_counts'})
        row['eval_ticket_counts'] = eval_result['ticket_counts']
    history.append(row)
    pd.DataFrame(history).to_csv(history_path, index=False)

    msg = (
        f"update {update:03d} | trans={len(transitions)} "
        f"rollout_wr={rollout_wr*100:.1f}% turn={rollout_turn:.1f} "
        f"loss={train_metrics.get('loss', 0):.4f} kl={train_metrics.get('approx_kl', 0):.4f} "
        f"entropy={train_metrics.get('entropy', 0):.3f}"
    )
    if eval_result is not None:
        msg += f" | eval_wr={eval_result['mrx_winrate']*100:.1f}% score={eval_result['validation_score']:.1f}"
    print(msg)

final_eval = eval_argmax(EVAL_GAMES, seed_offset=SEED + 900000)
torch.save(checkpoint_payload(UPDATES, final_eval, history), last_path)
pd.DataFrame(history).to_csv(history_path, index=False)

print('Best checkpoint:', best_path)
print('Last checkpoint:', last_path)
print('History:', history_path)
print('Final eval:', json.dumps(final_eval, indent=2))
print(f'Elapsed: {(time.time() - started) / 60:.1f} min')


## 8. Curves

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].plot(hist_df['update'], hist_df['rollout_mrx_wr'] * 100, label='rollout')
if 'eval_mrx_winrate' in hist_df:
    eval_df = hist_df.dropna(subset=['eval_mrx_winrate'])
    axes[0].plot(eval_df['update'], eval_df['eval_mrx_winrate'] * 100, 'o-', label='eval argmax')
axes[0].set_title('Mr.X winrate')
axes[0].set_ylabel('%')
axes[0].legend()

axes[1].plot(hist_df['update'], hist_df['train_loss'], label='loss')
axes[1].plot(hist_df['update'], hist_df['train_policy_loss'], label='policy')
axes[1].plot(hist_df['update'], hist_df['train_value_loss'], label='value')
axes[1].set_title('PPO losses')
axes[1].legend()

axes[2].plot(hist_df['update'], hist_df['train_entropy'], label='entropy')
axes[2].plot(hist_df['update'], hist_df['train_approx_kl'], label='approx KL')
axes[2].plot(hist_df['update'], hist_df['train_clipfrac'], label='clipfrac')
axes[2].set_title('Policy diagnostics')
axes[2].legend()

plt.tight_layout()
curves_path = OUT_DIR / f'mrx_rgnn_ppo_curves_{RUN_TAG}.png'
plt.savefig(curves_path, dpi=160)
plt.show()

print('Saved curves:', curves_path)
print('All outputs:')
for p in sorted(OUT_DIR.glob(f'*{RUN_TAG}*')):
    print(' ', p, p.stat().st_size)
